In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import joblib  # For saving models
import time  # For measuring computational time

# Load the dataset
data = pd.read_csv("/Synthetic Frailty Data.csv")  # Replace with your dataset's file path
data.head()

# Preprocess the data
label_encoder = LabelEncoder()

# Map 'Functional Reach Result' to numeric values
data['Functional Reach Result'] = label_encoder.fit_transform(data['Functional Reach Result'])

# Define input features (X) and output target (y)
X = data[['Functional Reach Test (cm)']]
y = data['Functional Reach Result']

# Standardize features for models like SVM and KNN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Save the scaler and label encoder
joblib.dump(scaler, 'Scaler_FRT.pkl')
print("Scaler saved as 'Scaler_FRT.pkl'")
joblib.dump(label_encoder, 'Label_Encoder_FRT.pkl')  # Save the LabelEncoder
print("Label Encoder saved as 'Label_Encoder_FRT.pkl'")

# Initialize models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# Train and evaluate models
results = {}
computation_costs = {}

for model_name, model in models.items():
    start_time = time.time()  # Start timing
    model.fit(X_train, y_train)
    end_time = time.time()  # End timing

    # Predictions
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Computational cost (training time in seconds)
    training_time = end_time - start_time
    computation_costs[model_name] = training_time

    # Print model results
    print(f"Model: {model_name}")
    print(f"Accuracy: {acc:.2f}")
    print(f"Training Time (seconds): {training_time:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=[str(cls) for cls in label_encoder.classes_]))

    print("-" * 50)
    results[model_name] = acc

    # Save the model as a .pkl file with _FRT suffix
    file_name = f"{model_name.replace(' ', '_')}_FRT.pkl"  # Append _FRT to the file name
    joblib.dump(model, file_name)
    print(f"Model '{model_name}' saved as '{file_name}'")

# Compare accuracies and computational costs
print("\nModel Performance Summary:")
for model_name, acc in results.items():
    print(f"{model_name}: Accuracy = {acc:.2f}, Training Time = {computation_costs[model_name]:.4f} seconds")


Scaler saved as 'Scaler_FRT.pkl'
Label Encoder saved as 'Label_Encoder_FRT.pkl'
Model: Logistic Regression
Accuracy: 1.00
Training Time (seconds): 0.0074

Classification Report:
               precision    recall  f1-score   support

    High Risk       1.00      1.00      1.00        47
Moderate Risk       1.00      1.00      1.00        51
         Pass       1.00      1.00      1.00       102

     accuracy                           1.00       200
    macro avg       1.00      1.00      1.00       200
 weighted avg       1.00      1.00      1.00       200

--------------------------------------------------
Model 'Logistic Regression' saved as 'Logistic_Regression_FRT.pkl'
Model: Random Forest
Accuracy: 1.00
Training Time (seconds): 0.1667

Classification Report:
               precision    recall  f1-score   support

    High Risk       1.00      1.00      1.00        47
Moderate Risk       1.00      1.00      1.00        51
         Pass       1.00      1.00      1.00       102

  

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [04:11:49] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [9]:
import os
print(os.path.getsize("K-Nearest_Neighbors_FRT.pkl"))  # Check file size in bytes


28550


In [8]:
import shutil

# List of files to download
files_to_download = ["K-Nearest_Neighbors_FRT.pkl", "Scaler_FRT.pkl", "Label_Encoder_FRT.pkl"]

# Copy files to a zip folder
shutil.make_archive("FRT_Models", 'zip', '.', base_dir=None)

print("Download the 'FRT_Models.zip' file from your directory.")


Download the 'FRT_Models.zip' file from your directory.


In [10]:
import sklearn
print(sklearn.__version__)

1.6.1
